# Receipt Field Extraction with LoRA

I wanted to get hands-on with fine-tuning an open-weight LLM instead of just calling an API, so
I picked a small, concrete task: given raw receipt text (with some OCR-style noise thrown in),
extract the vendor, date, line items, and total as clean JSON.

Steps:
1. Generate synthetic receipt data myself (with injected noise, so it's not a trivial clean dataset)
2. Evaluate the base model before touching it
3. Fine-tune with LoRA (PEFT)
4. Build my own field-level evaluation harness (not just loss — actual per-field accuracy)
5. Compare before vs after

Runs in **Google Colab** with a GPU runtime (Runtime > Change runtime type > T4 GPU).
Takes about 20-30 minutes on a free T4.

In [ ]:
# 1. Setup
!pip install -q transformers peft accelerate datasets bitsandbytes trl


In [ ]:
!pip uninstall -y torchao
!pip install -q transformers peft accelerate datasets bitsandbytes trl

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import torch, random, json, re
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


## Generating my own data

Instead of downloading a dataset, I wrote a generator that creates fake receipts with random vendors, items, and prices, then messes up the text a bit (swapped characters, extra spaces) to mimic real OCR output. This way I control exactly how hard the task is.

In [ ]:
VENDORS = ["Cafe Aroma", "Green Grocers", "TechMart Electronics", "Bloom Florist",
           "Urban Bites Diner", "Swift Pharmacy", "Northside Hardware", "Blue Bottle Coffee",
           "Fresh Basket Market", "Metro Bookstore"]

ITEMS_POOL = ["Coffee", "Sandwich", "Notebook", "USB Cable", "Bread Loaf", "Milk 1L",
              "Painkillers", "Screwdriver Set", "Flower Bouquet", "Paperback Novel",
              "Salad Bowl", "Water Bottle", "Charger", "Batteries", "Tea Bags"]

def random_date():
    day = random.randint(1, 28)
    month = random.randint(1, 12)
    year = random.choice([2024, 2025, 2026])
    return f"{day:02d}/{month:02d}/{year}"

def inject_noise(text):
    """Simulate common OCR artifacts."""
    text = text.replace("O", "0") if random.random() < 0.15 else text
    if random.random() < 0.2:
        text = text.replace(" ", "  ", 1)  # occasional double space
    if random.random() < 0.15:
        text = text.replace("$", "S")  # OCR confusing $ for S
    return text

def generate_receipt():
    vendor = random.choice(VENDORS)
    date = random_date()
    n_items = random.randint(2, 5)
    chosen_items = random.sample(ITEMS_POOL, n_items)
    line_items = []
    total = 0.0
    lines_text = []
    for item in chosen_items:
        qty = random.randint(1, 3)
        price = round(random.uniform(1.5, 40.0), 2)
        line_total = round(qty * price, 2)
        total += line_total
        line_items.append({"item": item, "qty": qty, "price": price})
        lines_text.append(f"{qty} x {item} ... ${price:.2f}")
    total = round(total, 2)

    receipt_text = f"{vendor}\nDate: {date}\n" + "\n".join(lines_text) + f"\nTOTAL: ${total:.2f}"
    receipt_text = inject_noise(receipt_text)

    target = {
        "vendor": vendor,
        "date": date,
        "items": line_items,
        "total": total
    }
    return receipt_text, target

# quick sanity check
r, t = generate_receipt()
print(r)
print()
print(json.dumps(t, indent=2))


Green Grocers
Date: 01/12/2025
1 x USB Cable ... $27.55
3 x Notebook ... $4.85
2 x Water Bottle ... $2.72
T0TAL: $47.54

{
  "vendor": "Green Grocers",
  "date": "01/12/2025",
  "items": [
    {
      "item": "USB Cable",
      "qty": 1,
      "price": 27.55
    },
    {
      "item": "Notebook",
      "qty": 3,
      "price": 4.85
    },
    {
      "item": "Water Bottle",
      "qty": 2,
      "price": 2.72
    }
  ],
  "total": 47.54
}


In [ ]:
def build_dataset(n):
    rows = []
    for _ in range(n):
        text, target = generate_receipt()
        prompt = (
            "Extract the vendor, date, items (with qty and price), and total from this "
            "receipt as JSON with keys vendor, date, items, total.\n\nReceipt:\n" + text
        )
        completion = json.dumps(target)
        rows.append({"prompt": prompt, "completion": completion})
    return rows

train_rows = build_dataset(400)
val_rows = build_dataset(60)
test_rows = build_dataset(60)

print(len(train_rows), len(val_rows), len(test_rows))


400 60 60


## Loading the base model

Using `Qwen/Qwen2.5-0.5B-Instruct` — small enough to fine-tune on a free Colab GPU, and already instruction-tuned, so LoRA is steering its output format rather than teaching it to follow instructions from scratch.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## Baseline eval (before touching the model)

Before fine-tuning anything, I need a "before" number to know if the fine-tuning actually did anything. I check field by field — vendor, date, total, items — not just whether it output valid JSON.

In [ ]:
def format_prompt(prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_completion(model, prompt_text, max_new_tokens=300):
    formatted = format_prompt(prompt_text)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text

def try_parse_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

def score_prediction(pred, target):
    """Returns dict of field-level correctness (0/1) plus overall exact match."""
    if pred is None:
        return {"valid_json": 0, "vendor": 0, "date": 0, "total": 0, "items": 0}
    scores = {"valid_json": 1}
    scores["vendor"] = int(pred.get("vendor", "").strip().lower() == target["vendor"].strip().lower())
    scores["date"] = int(pred.get("date", "").strip() == target["date"].strip())
    try:
        scores["total"] = int(abs(float(pred.get("total", -1e9)) - target["total"]) < 0.01)
    except (TypeError, ValueError):
        scores["total"] = 0
    scores["items"] = int(pred.get("items") == target["items"])
    return scores

def evaluate(model, rows, n=30, verbose_examples=0):
    agg = {"valid_json": 0, "vendor": 0, "date": 0, "total": 0, "items": 0}
    shown = 0
    for row in rows[:n]:
        raw = generate_completion(model, row["prompt"])
        pred = try_parse_json(raw)
        target = json.loads(row["completion"])
        scores = score_prediction(pred, target)
        for k in agg:
            agg[k] += scores[k]
        if shown < verbose_examples:
            print("---- Example ----")
            print("Model output:", raw[:300])
            print("Target:", row["completion"])
            print("Scores:", scores)
            shown += 1
    n_eval = min(n, len(rows))
    return {k: v / n_eval for k, v in agg.items()}

print("Evaluating BASE model (before fine-tuning)...")
base_scores = evaluate(base_model, test_rows, n=30, verbose_examples=2)
print("Base model field-level accuracy:", base_scores)


Evaluating BASE model (before fine-tuning)...
---- Example ----
Model output: ```json
{
  "vendor": "Bloom Florist",
  "date": "28/01/2024",
  "items": [
    {
      "name": "Salad Bowl",
      "qty": 3,
      "price": 34.54
    },
    {
      "name": "Notebook",
      "qty": 3,
      "price": 8.85
    },
    {
      "name": "Tea Bags",
      "qty": 2,
      "price": 31.13
  
Target: {"vendor": "Bloom Florist", "date": "28/01/2024", "items": [{"item": "Salad Bowl", "qty": 3, "price": 34.54}, {"item": "Notebook", "qty": 3, "price": 8.85}, {"item": "Tea Bags", "qty": 2, "price": 31.13}, {"item": "USB Cable", "qty": 1, "price": 19.75}, {"item": "Painkillers", "qty": 1, "price": 5.98}], "total": 218.16}
Scores: {'valid_json': 1, 'vendor': 1, 'date': 1, 'total': 1, 'items': 0}
---- Example ----
Model output: ```json
{
  "vendor": "Blue Bottle Coffee",
  "date": "16/01/2025",
  "items": [
    {
      "name": "USB Cable",
      "qty": 2,
      "price": 3.46
    },
    {
      "name": "Bread L

## LoRA fine-tuning

Standard PEFT LoRA config, targeting the attention projection layers. Kept small (few epochs, small rank) since the task is narrow and the base model already knows how to follow instructions — I'm just nudging its output format/reliability.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [ ]:
def to_training_text(row):
    formatted_prompt = format_prompt(row["prompt"])
    full_text = formatted_prompt + row["completion"] + tokenizer.eos_token
    return {"text": full_text}

train_ds = Dataset.from_list(train_rows).map(to_training_text)
val_ds = Dataset.from_list(val_rows).map(to_training_text)

def tokenize_fn(examples):
    out = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./lora-receipt-extractor",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    bf16=torch.cuda.is_available(),
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,0.343278,0.315226
2,0.271301,0.266362
3,0.262469,0.260610


TrainOutput(global_step=150, training_loss=0.4222392288843791, metrics={'train_runtime': 758.2085, 'train_samples_per_second': 1.583, 'train_steps_per_second': 0.198, 'total_flos': 1323341866598400.0, 'train_loss': 0.4222392288843791, 'epoch': 3.0})

## Eval after fine-tuning

Same field-level check as before, on the same held-out test set, so it's a fair comparison.

In [ ]:
print("Evaluating FINE-TUNED model...")
model.eval()
finetuned_scores = evaluate(model, test_rows, n=30, verbose_examples=3)

print("\n===== RESULTS =====")
print("Base model :", base_scores)
print("Fine-tuned :", finetuned_scores)


Evaluating FINE-TUNED model...
---- Example ----
Model output: {"vendor": "Bloom Florist", "date": "28/01/2024", "items": [{"item": "Salad Bowl", "qty": 3, "price": 34.54}, {"item": "Notebook", "qty": 3, "price": 8.85}, {"item": "Tea Bags", "qty": 2, "price": 31.13}, {"item": "USB Cable", "qty": 1, "price": 19.75}, {"item": "Painkillers", "qty": 1, "price": 5.9
Target: {"vendor": "Bloom Florist", "date": "28/01/2024", "items": [{"item": "Salad Bowl", "qty": 3, "price": 34.54}, {"item": "Notebook", "qty": 3, "price": 8.85}, {"item": "Tea Bags", "qty": 2, "price": 31.13}, {"item": "USB Cable", "qty": 1, "price": 19.75}, {"item": "Painkillers", "qty": 1, "price": 5.98}], "total": 218.16}
Scores: {'valid_json': 1, 'vendor': 1, 'date': 1, 'total': 1, 'items': 1}
---- Example ----
Model output: {"vendor": "Blue Bottle Coffee", "date": "16/01/2025", "items": [{"item": "USB Cable", "qty": 2, "price": 3.46}, {"item": "Bread Loaf", "qty": 1, "price": 5.45}, {"item": "Paperback Novel", "qty": 3, 

## Saving results

In [ ]:
results = {
    "model": MODEL_NAME,
    "lora_config": {"r": 8, "alpha": 16, "target_modules": ["q_proj","v_proj","k_proj","o_proj"]},
    "train_size": len(train_rows),
    "val_size": len(val_rows),
    "test_size": len(test_rows),
    "base_model_scores": base_scores,
    "finetuned_model_scores": finetuned_scores,
}
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))


{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "lora_config": {
    "r": 8,
    "alpha": 16,
    "target_modules": [
      "q_proj",
      "v_proj",
      "k_proj",
      "o_proj"
    ]
  },
  "train_size": 400,
  "val_size": 60,
  "test_size": 60,
  "base_model_scores": {
    "valid_json": 1.0,
    "vendor": 0.9,
    "date": 0.9666666666666667,
    "total": 1.0,
    "items": 0.0
  },
  "finetuned_model_scores": {
    "valid_json": 1.0,
    "vendor": 1.0,
    "date": 1.0,
    "total": 1.0,
    "items": 1.0
  }
}


In [ ]:
# Save the LoRA adapter
model.save_pretrained("lora-receipt-extractor-adapter")
tokenizer.save_pretrained("lora-receipt-extractor-adapter")
print("Saved adapter to ./lora-receipt-extractor-adapter")


Saved adapter to ./lora-receipt-extractor-adapter
